In [0]:
-- Day6 regression validation notebook (manual QA gate)
-- Scope: direct 1d pipeline, macro target in enterprise.gold_ref
-- IMPORTANT: this notebook is for validation only, not production ETL.

-- COMMAND ----------

-- A. Table existence check
WITH expected AS (
  SELECT 'enterprise' AS catalog, 'bronze_market' AS schema, 'crypto_ohlc_raw' AS table_name UNION ALL
  SELECT 'enterprise', 'bronze_ref', 'ecb_fx_raw' UNION ALL
  SELECT 'enterprise', 'bronze_ref', 'fred_series_raw' UNION ALL
  SELECT 'enterprise', 'silver_market', 'crypto_ohlc_1d' UNION ALL
  SELECT 'enterprise', 'silver_ref', 'ecb_fx_daily' UNION ALL
  SELECT 'enterprise', 'silver_ref', 'fred_series_daily' UNION ALL
  SELECT 'enterprise', 'gold_market', 'ohlc_1d' UNION ALL
  SELECT 'enterprise', 'gold_ref', 'market_macro_daily' UNION ALL
  SELECT 'enterprise', 'gold_observability', 'pipeline_metrics_daily'
)
SELECT
  e.catalog,
  e.schema,
  e.table_name,
  CASE WHEN t.table_name IS NULL THEN 'MISSING' ELSE 'OK' END AS status
FROM expected e
LEFT JOIN system.information_schema.tables t
  ON t.table_catalog = e.catalog
 AND t.table_schema = e.schema
 AND t.table_name = e.table_name
ORDER BY e.catalog, e.schema, e.table_name;


In [0]:
-- B1. Row count (target schema path; this will fail if gold_ref table is not materialized yet)
SELECT 'enterprise.bronze_market.crypto_ohlc_raw' AS table_name, COUNT(*) AS row_cnt FROM enterprise.bronze_market.crypto_ohlc_raw
UNION ALL
SELECT 'enterprise.silver_market.crypto_ohlc_1d', COUNT(*) FROM enterprise.silver_market.crypto_ohlc_1d
UNION ALL
SELECT 'enterprise.silver_ref.ecb_fx_daily', COUNT(*) FROM enterprise.silver_ref.ecb_fx_daily
UNION ALL
SELECT 'enterprise.silver_ref.fred_series_daily', COUNT(*) FROM enterprise.silver_ref.fred_series_daily
UNION ALL
SELECT 'enterprise.gold_market.ohlc_1d', COUNT(*) FROM enterprise.gold_market.ohlc_1d
UNION ALL
SELECT 'enterprise.gold_ref.market_macro_daily', COUNT(*) FROM enterprise.gold_ref.market_macro_daily
UNION ALL
SELECT 'enterprise.gold_observability.pipeline_metrics_daily', COUNT(*) FROM enterprise.gold_observability.pipeline_metrics_daily;


In [0]:
-- B2. Legacy fallback row count (use only when gold_ref target table is missing)
SELECT 'enterprise.gold_market.market_macro_daily' AS table_name, COUNT(*) AS row_cnt
FROM enterprise.gold_market.market_macro_daily;


In [0]:
-- C. Freshness check (minutes)
SELECT 'bronze_crypto' AS table_name, MAX(ingestion_ts) AS max_ts,
       TIMESTAMPDIFF(MINUTE, MAX(ingestion_ts), CURRENT_TIMESTAMP()) AS freshness_min
FROM enterprise.bronze_market.crypto_ohlc_raw
UNION ALL
SELECT 'silver_crypto_1d', MAX(bar_end_ts), TIMESTAMPDIFF(MINUTE, MAX(bar_end_ts), CURRENT_TIMESTAMP())
FROM enterprise.silver_market.crypto_ohlc_1d
UNION ALL
SELECT 'silver_ecb_daily', CAST(MAX(rate_date) AS TIMESTAMP), TIMESTAMPDIFF(MINUTE, CAST(MAX(rate_date) AS TIMESTAMP), CURRENT_TIMESTAMP())
FROM enterprise.silver_ref.ecb_fx_daily
UNION ALL
SELECT 'silver_fred_daily', CAST(MAX(obs_date) AS TIMESTAMP), TIMESTAMPDIFF(MINUTE, CAST(MAX(obs_date) AS TIMESTAMP), CURRENT_TIMESTAMP())
FROM enterprise.silver_ref.fred_series_daily
UNION ALL
SELECT 'gold_ref_market_macro_daily', MAX(mart_ts), TIMESTAMPDIFF(MINUTE, MAX(mart_ts), CURRENT_TIMESTAMP())
FROM enterprise.gold_ref.market_macro_daily
UNION ALL
SELECT 'obs_pipeline_metrics_daily', MAX(computed_ts), TIMESTAMPDIFF(MINUTE, MAX(computed_ts), CURRENT_TIMESTAMP())
FROM enterprise.gold_observability.pipeline_metrics_daily;


In [0]:
-- D1. Duplicate key group count in silver crypto
SELECT COUNT(*) AS dup_group_cnt
FROM (
  SELECT source, symbol, bar_start_ts, COUNT(*) c
  FROM enterprise.silver_market.crypto_ohlc_1d
  GROUP BY source, symbol, bar_start_ts
  HAVING COUNT(*) > 1
);

-- COMMAND ----------

-- D2. Invalid OHLC row count in silver crypto
SELECT COUNT(*) AS bad_ohlc_rows
FROM enterprise.silver_market.crypto_ohlc_1d
WHERE open <= 0
   OR high <= 0
   OR low <= 0
   OR close <= 0
   OR high < GREATEST(open, close)
   OR low > LEAST(open, close);


In [0]:
-- E1. Macro null-rate check on target table
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN eurusd_rate IS NULL THEN 1 ELSE 0 END) / COUNT(*) AS null_rate_eurusd_rate,
  SUM(CASE WHEN fedfunds  IS NULL THEN 1 ELSE 0 END) / COUNT(*) AS null_rate_fedfunds,
  SUM(CASE WHEN close_px  IS NULL THEN 1 ELSE 0 END) / COUNT(*) AS null_rate_close_px
FROM enterprise.gold_ref.market_macro_daily;

-- COMMAND ----------

-- E2. Macro missing trade-day ratio (target table)
WITH m AS (
  SELECT DISTINCT trade_date
  FROM enterprise.gold_ref.market_macro_daily
),
missing AS (
  SELECT DISTINCT trade_date
  FROM enterprise.gold_ref.market_macro_daily
  WHERE eurusd_rate IS NULL OR fedfunds IS NULL
)
SELECT
  (SELECT COUNT(*) FROM m) AS total_trade_days,
  (SELECT COUNT(*) FROM missing) AS days_with_macro_missing,
  CAST((SELECT COUNT(*) FROM missing) AS DOUBLE) / NULLIF((SELECT COUNT(*) FROM m), 0) AS missing_day_ratio;


In [0]:
-- E3. Legacy fallback macro null-rate (if target table is not yet materialized)
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN eurusd_rate IS NULL THEN 1 ELSE 0 END) / COUNT(*) AS null_rate_eurusd_rate,
  SUM(CASE WHEN fedfunds  IS NULL THEN 1 ELSE 0 END) / COUNT(*) AS null_rate_fedfunds,
  SUM(CASE WHEN close_px  IS NULL THEN 1 ELSE 0 END) / COUNT(*) AS null_rate_close_px
FROM enterprise.gold_market.market_macro_daily;

-- COMMAND ----------

-- E4. Legacy fallback missing day ratio
WITH m AS (
  SELECT DISTINCT trade_date
  FROM enterprise.gold_market.market_macro_daily
),
missing AS (
  SELECT DISTINCT trade_date
  FROM enterprise.gold_market.market_macro_daily
  WHERE eurusd_rate IS NULL OR fedfunds IS NULL
)
SELECT
  (SELECT COUNT(*) FROM m) AS total_trade_days,
  (SELECT COUNT(*) FROM missing) AS days_with_macro_missing,
  CAST((SELECT COUNT(*) FROM missing) AS DOUBLE) / NULLIF((SELECT COUNT(*) FROM m), 0) AS missing_day_ratio;


In [0]:
-- F. Observability latest-day status summary
WITH d AS (
  SELECT MAX(metric_date) AS latest_metric_date
  FROM enterprise.gold_observability.pipeline_metrics_daily
)
SELECT
  metric_date,
  pipeline,
  table_name,
  row_count,
  freshness_minutes,
  null_rate,
  duplicate_rate,
  status
FROM enterprise.gold_observability.pipeline_metrics_daily
WHERE metric_date = (SELECT latest_metric_date FROM d)
ORDER BY status DESC, table_name;

-- COMMAND ----------

-- F2. Compact status summary
WITH d AS (
  SELECT MAX(metric_date) AS latest_metric_date
  FROM enterprise.gold_observability.pipeline_metrics_daily
)
SELECT status, COUNT(*) AS cnt
FROM enterprise.gold_observability.pipeline_metrics_daily
WHERE metric_date = (SELECT latest_metric_date FROM d)
GROUP BY status
ORDER BY status;
